In [145]:
import json

with open("aiod_model.json", "r") as f:
    conceptual_model = json.load(f)

In [146]:
for i in range(len(conceptual_model["classes"])):
    if conceptual_model["classes"][i]["name"] == "Event":
        print(i)

54


In [147]:
conceptual_model["classes"][54]["inherited_properties"][0]
conceptual_model["classes"][54]["direct_properties"][0]

{'name': 'organiser',
 'domain': ['Event'],
 'range': ['Agent'],
 'annotations': {'comment': 'The entity organising the Event.',
  'label': 'organiser'},
 'equivalent_properties': []}

Generate schema

In [149]:
# Jupyter only looks in its default paths (sys.path does not automatically include sibling directories).
import sys
import os

# Define the absolute path to `aiod/src`
SRC_PATH = os.path.abspath("../src")

# Add it to Python's module search path if it's not already there
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print(f"Added {SRC_PATH} to sys.path")

Added /home/taniya_das/Documents/AIOD-rest-api/src to sys.path


In [150]:
# Example schema generation
from database.model.ai_asset.ai_asset import AIAssetBase

json.loads(AIAssetBase.schema_json())
AIAssetBase.schema_json()

'{"title": "AIAssetBase", "type": "object", "properties": {"platform": {"title": "Platform", "description": "The external platform from which this resource originates. Leave empty if this item originates from AIoD. If platform is not None, the platform_resource_identifier should be set as well.", "maxLength": 64, "example": "example", "type": "string"}, "platform_resource_identifier": {"title": "Platform Resource Identifier", "description": "A unique identifier issued by the external platform that\'s specified in \'platform\'. Leave empty if this item is not part of an external platform. For example, for HuggingFace, this should be the <namespace>/<dataset_name>, and for Openml, the OpenML identifier.", "maxLength": 256, "example": "1", "type": "string"}, "name": {"title": "Name", "maxLength": 256, "example": "The name of this resource", "type": "string"}, "date_published": {"title": "Date Published", "description": "The datetime (utc) on which this resource was first published on an e

In [151]:
import os
import json
import importlib.util
from pathlib import Path
from pydantic import BaseModel
from sqlmodel import SQLModel
import pandas as pd

In [185]:
"""
Step 1:
Extracts JSON schema from Pydantic/SQLModel classes and records parent classes.
- Handles errors per module.
"""

"""
Step 1:
Extracts JSON schema from Pydantic/SQLModel classes and records parent classes.
- Handles errors per module.
- Adds relationships dynamically.
"""

# Base directory
BASE_DIR = "database/model"
OUTPUT_FILE = "schemas.json"


"""
Helper function
    to extract SQLModel.Relationship 
"""


def extract_relationships(model_class):
    """
    Extracts relationships dynamically for SQLModel classes.
    Returns a dictionary with relationships.
    """
    relationships = {}

    if hasattr(model_class, "__sqlmodel_relationships__"):
        for rel_name, rel_obj in model_class.__sqlmodel_relationships__.items():
            # Store the relationship as a property
            relationships[rel_name] = {"name": rel_name}

    return relationships


def get_initial_schema():
    """
    Extracts JSON schema from Pydantic/SQLModel classes and records parent classes.
    Also adds relationships as properties.
    """
    all_schemas = {}
    read_error = {}
    class_parents = {}

    for root, _, files in os.walk(os.path.join(SRC_PATH, BASE_DIR)):
        for file in files:
            if file.endswith(".py") and not file.startswith("__"):
                module_path = os.path.join(root, file)
                module_name = Path(module_path).with_suffix("").as_posix().split("/")[-1]

                try:
                    spec = importlib.util.spec_from_file_location(module_name, module_path)
                    module = importlib.util.module_from_spec(spec)
                    spec.loader.exec_module(module)

                    for attr_name in dir(module):
                        SQLModel.metadata.clear()
                        attr = getattr(module, attr_name)

                        # Only process Pydantic/SQLModel classes
                        if (
                            isinstance(attr, type)
                            and issubclass(attr, BaseModel)
                            and attr is not BaseModel
                        ):
                            schema = attr.schema_json()
                            schema_dict = json.loads(schema)

                            # Extract relationships
                            relationships = extract_relationships(attr)

                            # Add relationships to properties
                            schema_dict.setdefault("properties", {}).update(relationships)

                            # Store schema
                            all_schemas[attr.__name__] = schema_dict

                            # Get parent classes dynamically
                            parent_classes = [
                                parent.__name__
                                for parent in attr.__bases__
                                if isinstance(parent, type)
                                and issubclass(parent, BaseModel)
                                and parent is not BaseModel
                            ]

                            # Store parent mapping
                            class_parents[attr.__name__] = parent_classes

                except Exception as e:
                    print(f"Error processing {module_name}: {e}")
                    read_error.update({module_name: e})

    return all_schemas, read_error, class_parents


# Run function
# all_schemas, read_error, class_parents = get_initial_schema()

Error processing location: issubclass() arg 1 must be a class


/home/taniya_das/Documents/AIOD-rest-api/venv/lib/python3.11/site-packages/sqlmodel/main.py:638: SAWarning: This declarative base already contains a class with the same class name and module name as database.model.helper_functions.LinkTable, and will be replaced in the string-lookup table.
  DeclarativeMeta.__init__(cls, classname, bases, dict_, **kw)
/home/taniya_das/Documents/AIOD-rest-api/venv/lib/python3.11/site-packages/sqlmodel/main.py:638: SAWarning: This declarative base already contains a class with the same class name and module name as database.model.ai_asset.distribution.DistributionORM, and will be replaced in the string-lookup table.
  DeclarativeMeta.__init__(cls, classname, bases, dict_, **kw)
/home/taniya_das/Documents/AIOD-rest-api/venv/lib/python3.11/site-packages/sqlmodel/main.py:638: SAWarning: This declarative base already contains a class with the same class name and module name as database.model.ai_resource.note.NoteORM, and will be replaced in the string-lookup

In [154]:
"""
Step 2:
Recursively converts all dictionary keys to lowercase.
Ensures "title" and "properties" fields are in lowercase.
"""


def convert_to_lowercase(data):
    """
    Convert dictionary keys and relevant values to lowercase without recursion.
    Works for:
    - class_parents (keys & values)
    - all_schemas (keys, titles, property names, and required fields)
    """
    if isinstance(data, dict):
        lowercased_data = {}

        for key, value in data.items():
            lower_key = key.lower()

            # Handle different structures (list for class_parents, dict for all_schemas)
            if isinstance(value, list):
                lowercased_data[lower_key] = [v.lower() for v in value]  # Convert list values
            elif isinstance(value, dict):
                lowercased_schema = {
                    "name": value.get("title", lower_key).lower(),  # Convert title
                    "description": value.get("description", ""),
                    "type": value.get("type", "").lower(),
                    "properties": {},
                    "required": [
                        field.lower() for field in value.get("required", [])
                    ],  # Lowercase required fields
                }

                # Convert properties inside schema
                if "properties" in value:
                    for prop_name, prop_details in value["properties"].items():
                        lower_prop_name = prop_name.lower()
                        lowercased_schema["properties"][lower_prop_name] = {
                            "name": prop_details.get(
                                "title", lower_prop_name
                            ).lower(),  # Convert property title
                            "title": prop_details.get(
                                "title", lower_prop_name
                            ).lower(),  # Ensure title is lowercase
                            **{
                                k: v for k, v in prop_details.items() if k != "title"
                            },  # Copy other attributes
                        }

                lowercased_data[lower_key] = lowercased_schema

        return lowercased_data

    return data  # Return as is if not a dict

In [155]:
"""
Step 3:
Expands class_parents dictionary and accounts for all indirect parents. 
all_inheritance stores complete parent hierarchy.
"""


def get_complete_inheritance(class_parents):
    all_inheritance = {
        key: set(value) for key, value in class_parents.items()
    }  # sets to avoid duplicates

    # Keep iterating until all inherited classes are fully expanded
    for key in class_parents:
        queue = list(class_parents[key])  # Start with direct parents

        while queue:
            parent = queue.pop(0)
            if parent in class_parents:
                new_parents = class_parents[parent]  # Get parent's parents
                queue.extend(new_parents)  # Add them to queue for further processing
                all_inheritance[key].update(new_parents)  # Add to final set

    # Convert sets back to lists for JSON compatibility
    all_inheritance = {k: list(v) for k, v in all_inheritance.items()}

    return all_inheritance

In [156]:
"""
Step 4:
Add inherited classes field to `properties` of child class. 
Ensures no duplicates
"""


def add_inherited_properties(generated_schema, all_inheritance):
    """
    Adds inherited properties directly into 'properties' for each entity.
    """
    updated_schema = {}

    for entity_name, schema in generated_schema.items():
        updated_schema[entity_name] = schema
        # Copy the existing properties
        merged_properties = schema.get("properties")

        # If the entity has parent classes, inherit their properties
        for parent_class in all_inheritance.get(entity_name, []):
            # print(entity_name, parent_class)
            if parent_class in generated_schema:
                parent_properties = generated_schema[parent_class]["properties"]

                # Add missing properties from the parent class
                for prop_name, prop_details in parent_properties.items():
                    if prop_name not in merged_properties:  # Avoid duplicates
                        merged_properties[prop_name] = prop_details

        # Store the updated entity schema
        # updated_schema[entity_name] = schema
        updated_schema[entity_name]["properties"] = (
            merged_properties  # Update properties with inherited ones
        )

    return updated_schema

In [194]:
"""
Step 5:
Merges Base, Table, ORM, Type, Link classes into their main class.
Returns a new schema with merged properties.
"""


import re


def merge_base_classes(schema_with_inherited):
    merged_schema = {}
    class_mappings = {}

    # Identify base-like classes and map them to their main class
    for entity_name in schema_with_inherited:
        match = re.match(
            r"^(abstract)?(.+?)(base|table|orm|type|link)?$", entity_name, re.IGNORECASE
        )
        if match:
            core_name = match.group(2)  # The actual entity name
            main_class = core_name if core_name else entity_name
            class_mappings[entity_name] = main_class

    # Merge properties into main classes
    for entity_name, schema in schema_with_inherited.items():
        # normalized_name = entity_name
        main_entity_name = class_mappings.get(entity_name, entity_name)  # Resolve to main class
        # print(main_entity_name)

        if main_entity_name not in merged_schema:
            merged_schema[main_entity_name] = {
                "name": main_entity_name,
                "equivalent_classes": [],
                "properties": {},  # Will be renamed from "properties"
                "required": schema.get("required", []),
            }

        # Copy properties into direct_properties
        for prop_name, prop_details in schema.get("properties", {}).items():
            processed_property = prop_details
            processed_property["name"] = prop_name  # Rename "title" to "name"
            prop_name_cleaned = processed_property["name"]
            if "title" in processed_property:  # Remove "title" field if it exists
                del processed_property["title"]

            if prop_name_cleaned not in merged_schema[main_entity_name]["properties"]:
                merged_schema[main_entity_name]["properties"][prop_name_cleaned] = (
                    processed_property
                )

        # Merge required fields
        merged_schema[main_entity_name]["required"] = list(
            set(merged_schema[main_entity_name]["required"]) | set(schema.get("required", []))
        )

    return merged_schema

In [158]:
"""
Compare the conceptual model and implemented model
"""


def compare_schemas(conceptual_model, generated_schema):
    """
    Compare conceptual and generated schemas:
    - Identify missing and extra entities
    - Identify missing and extra properties for matching entities
    """
    # Convert both to case-insensitive dictionaries for easy lookup
    conceptual_dict = {entity["name"].lower(): entity for entity in conceptual_model}
    # generated_dict = {entity["name"].lower(): entity for entity in generated_schema}
    generated_dict = {key.lower(): value for key, value in generated_schema.items()}

    # Entities present in both schemas
    matching_entities = set(conceptual_dict.keys()) & set(generated_dict.keys())

    # Entities missing in generated schema
    missing_entities = set(conceptual_dict.keys()) - set(generated_dict.keys())

    # Entities extra in generated schema (not in conceptual model)
    extra_entities = set(generated_dict.keys()) - set(conceptual_dict.keys())

    # Compare properties for matching entities
    missing_properties = {}
    extra_properties = {}

    for entity_name in matching_entities:
        # Get conceptual model properties (merge direct & inherited properties)
        conceptual_properties = set(
            prop["name"].lower() for prop in conceptual_dict[entity_name]["direct_properties"]
        ) | set(
            prop["name"].lower() for prop in conceptual_dict[entity_name]["inherited_properties"]
        )

        # # Get generated schema properties
        # generated_properties = set(
        #     prop["name"].lower() for prop in generated_dict[entity_name]["direct_properties"]
        # )

        # Get generated schema properties
        generated_properties = set()

        if entity_name in generated_dict and "properties" in generated_dict[entity_name]:
            generated_properties = {
                generated_dict[entity_name]["properties"][prop]["name"].lower()
                for prop in generated_dict[entity_name]["properties"]
            }

        # Identify missing and extra properties
        missing_props = conceptual_properties - generated_properties
        extra_props = generated_properties - conceptual_properties

        if missing_props:
            missing_properties[entity_name] = list(missing_props)

        if extra_props:
            extra_properties[entity_name] = list(extra_props)

    # Store results
    comparison_results = {
        "matching_entities": list(matching_entities),
        "missing_entities": list(missing_entities),
        "extra_entities": list(extra_entities),
        "missing_properties": missing_properties,
        "extra_properties": extra_properties,
    }

    return comparison_results

In [159]:
def restructure_properties_single_row(properties_dict):
    """
    Restructures the properties dictionary into a DataFrame where:
    - Each entity is a row.
    - All its missing/extra properties are stored in the same row as separate columns.

    Parameters:
    - properties_dict: Dictionary with entities as keys and list of properties as values.

    Returns:
    - Pandas DataFrame
    """
    reformatted_data = [
        {"Entity": entity, **{f"Property {i + 1}": prop for i, prop in enumerate(props)}}
        for entity, props in properties_dict.items()
    ]

    return pd.DataFrame(reformatted_data)

In [160]:
import pandas as pd


def comparison_results_to_dataframe(comparison_results):
    """
    Convert comparison results dictionary into multiple Pandas DataFrames.

    Returns:
    - missing_entities_df: DataFrame listing missing entities.
    - extra_entities_df: DataFrame listing extra entities.
    - missing_properties_df: DataFrame listing missing properties per entity.
    - extra_properties_df: DataFrame listing extra properties per entity.
    """
    # Convert missing and extra entities to DataFrames
    missing_entities_df = pd.DataFrame(
        comparison_results.get("missing_entities", []), columns=["Missing Entities"]
    )
    extra_entities_df = pd.DataFrame(
        comparison_results.get("extra_entities", []), columns=["Extra Entities"]
    )

    # # Convert missing properties to DataFrame
    missing_properties_df = restructure_properties_single_row(
        comparison_results.get("missing_properties", {})
    )

    # Convert extra properties to DataFrame
    extra_properties_df = restructure_properties_single_row(
        comparison_results.get("extra_properties", {})
    )

    return missing_entities_df, extra_entities_df, missing_properties_df, extra_properties_df

In [161]:
"""
Step 1: 
    - all_schemas: generate intial schema
    - class_parents: identify direct inherited classes
    - read_error: error while generating schema using json_schema() function
Step 2: 
    - convert all_schemas and class_parents to lowercase
Step 3:
    - all_inheritance: expand parent-class mapping to get all inherited classes
Step 4:
    - schema_with_inherited: add inherited properties to each class in all_schema
Step 5:
    - merged_schema: merge properties and required from base, table, orm, type, link and abstract. 
    - rename "title" to "name"
"""

'\nStep 1: \n    - all_schemas: generate intial schema\n    - class_parents: identify direct inherited classes\n    - read_error: error while generating schema using json_schema() function\nStep 2: \n    - convert all_schemas and class_parents to lowercase\nStep 3:\n    - all_inheritance: expand parent-class mapping to get all inherited classes\nStep 4:\n    - schema_with_inherited: add inherited properties to each class in all_schema\nStep 5:\n    - merged_schema: merge properties and required from base, table, orm, type, link and abstract. \n    - rename "title" to "name"\n'

In [ ]:
all_schemas, read_error, class_parents = get_initial_schema()

all_schemas = convert_to_lowercase(all_schemas)
class_parents = convert_to_lowercase(class_parents)

all_inheritance = get_complete_inheritance(class_parents)

schema_with_inherited = add_inherited_properties(all_schemas, all_inheritance)

merged_schema = merge_base_classes(schema_with_inherited)

# Run comparison
comparison_results = compare_schemas(conceptual_model["classes"], merged_schema)

missing_entities_df, extra_entities_df, missing_properties_df, extra_properties_df = (
    comparison_results_to_dataframe(comparison_results)
)

In [198]:
missing_properties_df[missing_properties_df["Entity"] == "knowledgeasset"].dropna(axis=1)

,Entity,Property 1,Property 2,Property 3,Property 4,Property 5,Property 6,Property 7,Property 8,Property 9,Property 10
22,knowledgeasset,has_part,sameas,applies,business_sector,licence,location,date_created,part_of,pricing,documented_in


In [199]:
# missing_entities_df
# extra_entities_df
missing_properties_df
# # extra_properties_df

,Entity,Property 1,Property 2,Property 3,Property 4,Property 5,Property 6,Property 7,Property 8,Property 9,...,Property 11,Property 12,Property 13,Property 14,Property 15,Property 16,Property 17,Property 18,Property 19,Property 20
0,agent,has_part,sameas,business_sector,location,contact_info,part_of,documented_in,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,person,has_part,sameas,business_sector,location,languages,contact_info,part_of,documented_in,member_of,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,researcharea,sameas,relevant_link,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,dataset,has_part,sameas,applies,business_sector,licence,location,date_created,variable_measured,part_of,...,documented_in,pid,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,distribution,sameas,identifier,has_computational_requirement,date_modified,relevant_link,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,publication,has_part,sameas,applies,has_isbn,business_sector,licence,location,date_created,has_issn,...,pricing,documented_in,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,project,has_part,sameas,business_sector,location,total_cost_euros,part_of,documented_in,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,aiodentry,sameas,entry_created,modified,relevant_link,content,entry_published,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,location,region,sameas,city,longitude,street,latitude,elevation_m,relevant_link,postal_code,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,airesource,sameas,business_sector,location,part_of,documented_in,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [189]:
# save the df in a csv

missing_properties_df.to_csv("missing_properties.csv", index=False)
missing_entities_df.to_csv("missing_entities.csv", index=False)
extra_entities_df.to_csv("extra_entities.csv", index=False)
extra_properties_df.to_csv("extra_properties.csv", index=False)